# `baseline_models.py` — Baseline Model Implementations

## Purpose
Implements the **comparison baselines** used to demonstrate that the full ClearView model (RoBERTa + GCN + Aspect Attention + Hybrid Loss) is meaningfully better than simpler alternatives.

## Baselines

| Class | Description | What it tests |
|-------|-------------|---------------|
| `PlainRoBERTa` | RoBERTa + single shared [CLS] head | Benefit of aspect-awareness |
| `CrossEntropyLossWrapper` | Same arch as full model + vanilla CE | Benefit of hybrid loss |
| `BERTBaseline` | BERT-base (not RoBERTa) | Encoder choice matters? |
| `TFIDFSVMBaseline` | TF-IDF features + LinearSVC per aspect | Deep learning vs. classical ML |

## Usage
Use the `create_baseline(name, config)` factory function — it's called by `experiment_runner.py` when it encounters a B-series experiment.

In [ ]:
import torch
import torch.nn as nn
from transformers import RobertaModel, BertModel
import os, sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

## B1: Plain RoBERTa
The simplest transformer baseline. Uses [CLS] to represent the whole text and a single shared head for all aspects — aspect-unaware.

In [ ]:
class PlainRoBERTa(nn.Module):
    """
    RoBERTa + shared [CLS] head. No aspect identity passed into the model.
    Establishes the baseline benefit that just fine-tuning RoBERTa gives.
    """
    def __init__(self, roberta_model='roberta-base', num_classes=3, dropout=0.1):
        super(PlainRoBERTa, self).__init__()
        self.roberta    = RobertaModel.from_pretrained(roberta_model)  # Pre-trained encoder
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_classes)  # 768 = RoBERTa hidden dim

    def forward(self, input_ids, attention_mask, aspect_id=None, edge_index=None, **kwargs):
        # aspect_id is intentionally ignored — this model doesn't use aspect information
        output   = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls      = output.last_hidden_state[:, 0, :]  # Take [CLS] token representation
        cls      = self.dropout(cls)
        return self.classifier(cls)                   # Predict all 3 classes from [CLS]

    def get_num_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

## B2: CrossEntropyLossWrapper
Not a different model — same architecture as the full model, but with standard Cross-Entropy loss instead of the Hybrid loss. This isolates the contribution of the custom loss functions.

In [ ]:
class CrossEntropyLossWrapper(nn.Module):
    """
    Wraps nn.CrossEntropyLoss with the same interface as AspectSpecificLossManager,
    so experiment_runner can call .compute_loss() identically for all experiments.
    """
    def __init__(self):
        super(CrossEntropyLossWrapper, self).__init__()
        self.criterion = nn.CrossEntropyLoss()  # Standard loss — no imbalance handling

    def compute_loss(self, predictions, targets, aspect_ids, aspect_names):
        """Identical signature to AspectSpecificLossManager.compute_loss() for compatibility."""
        loss     = self.criterion(predictions, targets)
        loss_val = loss.item()
        return loss, {'ce': loss_val, 'total': loss_val}  # match expected dict shape

## B3: BERT Baseline

In [ ]:
class BERTBaseline(nn.Module):
    """
    BERT-base-uncased with a single shared [CLS] classification head.
    Same as PlainRoBERTa but with BERT encoder — tests whether RoBERTa's extra
    pre-training (dynamic masking, larger corpus) gives a meaningful boost.
    """
    def __init__(self, bert_model='bert-base-uncased', num_classes=3, dropout=0.1):
        super(BERTBaseline, self).__init__()
        self.bert       = BertModel.from_pretrained(bert_model)  # BERT (not RoBERTa)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask, aspect_id=None, edge_index=None, **kwargs):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls    = output.last_hidden_state[:, 0, :]
        cls    = self.dropout(cls)
        return self.classifier(cls)

    def get_num_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

## B4: TF-IDF + LinearSVC
A classical machine learning baseline. One TF-IDF vectoriser + SVM pipeline per aspect. No GPU, no transformers — shows how much deep learning adds beyond traditional bag-of-words NLP.

In [ ]:
class TFIDFSVMBaseline:
    """
    Classical TF-IDF + LinearSVC baseline.
    One 3-class (neg/neu/pos) SVM pipeline per aspect.
    Uses sklearn — no GPU required.
    """
    def __init__(self, aspect_names, max_features=50000, ngram_range=(1, 2)):
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.svm import LinearSVC
        from sklearn.calibration import CalibratedClassifierCV  # Adds predict_proba to LinearSVC
        from sklearn.pipeline import Pipeline

        self.aspect_names = aspect_names
        self.pipelines    = {}  # One trained pipeline per aspect

        for aspect in aspect_names:
            self.pipelines[aspect] = Pipeline([
                ('tfidf', TfidfVectorizer(
                    max_features=max_features,   # Limit vocabulary size
                    ngram_range=ngram_range,      # Unigrams + bigrams
                    sublinear_tf=True,            # Apply log scaling to TF values
                    strip_accents='unicode',
                    analyzer='word',
                    min_df=2,                     # Ignore tokens appearing < 2 times
                )),
                ('clf', CalibratedClassifierCV(LinearSVC(
                    class_weight='balanced',      # Handles class imbalance at training
                    max_iter=2000,
                    C=1.0,                        # Regularisation strength
                ))),
            ])

    def fit(self, df, label_map):
        """Train one SVM pipeline per aspect on rows where that aspect is labelled."""
        for aspect in self.aspect_names:
            mask = df[aspect].notna()          # Only use rows that have a label for this aspect
            if mask.sum() == 0: continue
            X = df.loc[mask, 'data'].astype(str).tolist()
            y = df.loc[mask, aspect].map(lambda v: label_map.get(str(v).lower(), -1)).tolist()
            valid_pairs = [(x, lbl) for x, lbl in zip(X, y) if lbl != -1]
            if not valid_pairs: continue
            X_v, y_v = zip(*valid_pairs)
            print(f'  Training SVM for {aspect}: {len(X_v)} samples')
            self.pipelines[aspect].fit(X_v, y_v)

    def predict(self, texts, aspect):
        """Predict class labels (integers) for a list of texts for the given aspect."""
        return self.pipelines[aspect].predict(texts)

    def predict_proba(self, texts, aspect):
        """Returns probability estimates [neg, neu, pos] for the given aspect."""
        return self.pipelines[aspect].predict_proba(texts)

## Factory Function

In [ ]:
def create_baseline(baseline_name: str, config: dict):
    """
    Factory: returns the right baseline model for a given name.
    Called by experiment_runner when encountering B-series experiments.
    """
    num_classes = config['model']['num_classes']   # Always 3 (neg/neu/pos)
    dropout     = config['model']['dropout']
    aspects     = config['aspects']['names']

    if baseline_name == 'plain_roberta':
        return PlainRoBERTa(config['model']['roberta_model'], num_classes, dropout)

    elif baseline_name == 'ce_loss':
        from models.model import create_model
        return create_model(config)   # Full architecture, loss is replaced by CrossEntropyLossWrapper

    elif baseline_name == 'bert_base':
        return BERTBaseline('bert-base-uncased', num_classes, dropout)

    elif baseline_name == 'tfidf_svm':
        return TFIDFSVMBaseline(aspect_names=aspects)

    else:
        raise ValueError(f"Unknown baseline: '{baseline_name}'")